# FloodScope 3D: evaluate the discharge forecast on held-out basins

Uses a run directory from `train_colab.ipynb`. Every number here is on **basins the model never saw**.

- **NSE** (Nash-Sutcliffe): 1 is perfect, 0 is no better than the long-term mean.
- **KGE** (Kling-Gupta): 1 is perfect; about -0.41 equals the mean-flow benchmark.
- **Persistence** (tomorrow = today) is the baseline to beat at short leads.
- **GloFAS** is its *reanalysis* at the gauge (Open-Meteo Flood API), i.e. GloFAS driven by observed weather. Archived GloFAS *forecasts* need a Copernicus CDS account, so this comparison favours GloFAS.

In [ ]:
import glob
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import requests

from floodscope_ml.evaluate import (
    basin_series,
    load_test_results,
    score_basin,
    summarise,
    today_flow,
)

WORK = Path("/content/drive/MyDrive/floodscope")
RUN_DIR = sorted((WORK / "runs").glob("floodscope_forecast_lstm_*"))[-1]
PREPARED = Path("/content/prepared")
CARAVAN = Path(glob.glob("/content/caravan_raw/*/timeseries")[0]).parent
results = load_test_results(RUN_DIR)
test_basins = sorted(results)
print(RUN_DIR.name, len(test_basins), "held-out basins")

In [ ]:
scores = []
for b in test_basins:
    for row in score_basin(basin_series(results, b), today_q=today_flow(PREPARED, b)):
        scores.append({**row, "basin": b})
per_basin = pd.DataFrame(scores)
table = summarise(scores)
table.round(3)

In [ ]:
attrs = pd.concat(
    [
        pd.read_csv(f, index_col="gauge_id")
        for f in glob.glob(f"{CARAVAN}/attributes/*/attributes_other_*.csv")
    ]
)
sample = random.Random(0).sample(test_basins, min(50, len(test_basins)))
g = []
for b in sample:
    lat, lon = attrs.loc[b, "gauge_lat"], attrs.loc[b, "gauge_lon"]
    r = requests.get(
        "https://flood-api.open-meteo.com/v1/flood",
        params=dict(
            latitude=lat,
            longitude=lon,
            daily="river_discharge",
            start_date="1997-01-01",
            end_date="2019-12-31",
        ),
        timeout=120,
    ).json()
    q = pd.Series(
        r["daily"]["river_discharge"], index=pd.to_datetime(r["daily"]["time"]), dtype=float
    )
    g += score_basin(basin_series(results, b), glofas_q=q / q.loc["1997-01-01":"2010-09-30"].mean())
    time.sleep(2)  # Open-Meteo rate limits
glofas = summarise(g)
glofas.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharex=True)
for ax, m in zip(axes, ["nse", "kge"]):
    ax.plot(table.index, table[m], marker="o", label="FloodScope LSTM (all held-out)")
    ax.plot(table.index, table[f"{m}_persistence"], marker="s", ls="--", label="Persistence")
    ax.plot(
        glofas.index, glofas[f"{m}_glofas"], marker="^", ls=":", label="GloFAS reanalysis (sample)"
    )
    ax.set_title(f"Median {m.upper()} by lead time")
    ax.set_xlabel("Lead (days)")
    ax.grid(alpha=0.3)
axes[0].legend()
plt.tight_layout()

In [ ]:
# Spread across basins at day 1 and day 7: medians hide bad basins.
per_basin[per_basin.lead.isin([1.0, 7.0])].boxplot(column="nse", by="lead", figsize=(6, 4))
plt.ylim(-1, 1)
plt.suptitle("")
plt.title("NSE across held-out basins")